# Notebook 1 — IC to Basin Outlet · GEE/xee Workflow
### GeomorphConn · Singh, Cavalli & Crema (2026)

This notebook demonstrates a fully cloud-driven **IC-outlet** workflow using Google Earth Engine (GEE) + xee.

It covers:

1. Fetching DEM, NDVI, and rainfall layers from GEE for any region
2. Running IC with multiple weight configurations, including DEM-only roughness
3. Exporting IC outputs as GeoTIFFs for GIS workflows

### Weight options shown in this notebook
- DEM roughness only (no NDVI/rainfall inputs)
- Rainfall only
- NDVI only
- NDVI + Rainfall (default hydrological weighting)
- NDVI + Rainfall + DEM roughness

### Inputs
- Required: region constraint (BOUNDS) via tuple or basin polygon file
- Optional: no local rasters needed

### Output
GeoTIFFs of IC, W, S, Dup, Ddn for the selected scenario and comparison plots.

## 0 · Configuration — **edit here**

In [ ]:
# ─── Region constraint ───────────────────────────────────────────────────────
BOUNDS = (78.8, 29.6, 80.0, 30.5)
# You can also pass a polygon file (basin boundary) instead of tuple:
# BOUNDS = "data/basin_boundary.geojson"

# ─── GEE configuration ───────────────────────────────────────────────────────
GEE_PROJECT   = "drylands-aberuni"   # your GEE project
DEM_SOURCE    = "COPDEM30"              # SRTM | COPDEM30 | MERIT
RF_SOURCE     = "CHIRPS"               # CHIRPS | ERA5 | PERSIANN
NDVI_SOURCE   = "SENTINEL2"            # SENTINEL2 | LANDSAT8 | LANDSAT9
START_DATE    = "2020-05-01"
END_DATE      = "2020-09-30"
SCALE         = 30
CRS           = "EPSG:4326"

# ─── IC routing parameters ───────────────────────────────────────────────────
FLOW_DIRECTOR = "FlowDirectorDINF"     # D8 | FlowDirectorDINF | FlowDirectorMFD
W_MIN         = 0.005
USE_DEG_APPROX = True

# ─── Weight scenarios to evaluate ───────────────────────────────────────────
WEIGHT_SCENARIOS = [
    "roughness_only",
    "rainfall_only",
    "ndvi_only",
    "ndvi_rainfall",
    "ndvi_rainfall_roughness",
]

PLOT_SCENARIO = "ndvi_rainfall"

# ─── Output ──────────────────────────────────────────────────────────────────
from pathlib import Path
OUTPUT_DIR = Path("output_nb1")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1 · Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import rasterio

from landlab import RasterModelGrid
from geomorphconn.components import ConnectivityIndex
from geomorphconn.gee import GEEFetcher
from geomorphconn.weights import (
    WeightBuilder,
    NDVIWeight,
    RainfallWeight,
    preset_rainfall_ndvi,
    preset_rainfall_ndvi_roughness,
    preset_roughness_only,
)

import geomorphconn
print(f"GeomorphConn version: {geomorphconn.__version__}")

## 2 · Fetch data from Google Earth Engine

On first run this will open a browser window for GEE authentication.

In [ ]:
GEEFetcher.list_sources()   # print available datasets

In [ ]:
fetcher = GEEFetcher(
    bounds         = BOUNDS,
    dem_source     = DEM_SOURCE,
    rainfall_source= RF_SOURCE,
    ndvi_source    = NDVI_SOURCE,
    start_date     = START_DATE,
    end_date       = END_DATE,
    scale          = SCALE,
    gee_project    = GEE_PROJECT,
)

dem, ndvi, rainfall, profile = fetcher.fetch()
nrows, ncols = dem.shape
dx = abs(profile["transform"].a)
print(f"\nGrid: {nrows} × {ncols} cells, dx = {dx:.1f} m")

In [ ]:
# Quick preview of inputs
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=100)
for ax, arr, title, cmap in zip(
        axes,
        [dem, ndvi, rainfall],
        [f"{DEM_SOURCE} DEM (m)", f"NDVI ({NDVI_SOURCE})", f"Rainfall mm ({RF_SOURCE})"],
        ["terrain", "RdYlGn", "Blues"]):
    im = ax.imshow(arr, cmap=cmap, interpolation="nearest")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("GEE-fetched inputs", fontweight="bold")
plt.tight_layout(); plt.show()

## 3 · Build Landlab grid

In [ ]:
# GeoTIFF row 0 = north; Landlab row 0 = south → flipud before ravel
grid = RasterModelGrid((nrows, ncols), xy_spacing=dx)

dem_ll = np.flipud(dem).copy()
# Replace NaN elevations with a low fill value
dem_ll[np.isnan(dem_ll)] = np.nanmin(dem_ll) - 1.0

grid.add_field("topographic__elevation", dem_ll.ravel(), at="node")

ndvi_ll     = np.flipud(np.where(np.isnan(ndvi),     0.0, ndvi)).ravel()
rainfall_ll = np.flipud(np.where(np.isnan(rainfall), 0.0, rainfall)).ravel()

print(f"Grid: {grid.number_of_nodes:,} nodes")

## 4 · Run IC across weight scenarios

This demonstrates that IC can run with DEM-only roughness (no NDVI/rainfall) or any NDVI/rainfall combination.

In [ ]:
import time

def make_weight_builder(name, rainfall_nodes, ndvi_nodes, grid_obj):
    if name == "roughness_only":
        return preset_roughness_only(grid_obj)
    if name == "rainfall_only":
        return WeightBuilder().add(RainfallWeight(rainfall_nodes))
    if name == "ndvi_only":
        return WeightBuilder().add(NDVIWeight(ndvi_nodes))
    if name == "ndvi_rainfall":
        return preset_rainfall_ndvi(rainfall_nodes, ndvi_nodes)
    if name == "ndvi_rainfall_roughness":
        return preset_rainfall_ndvi_roughness(rainfall_nodes, ndvi_nodes, grid_obj)
    raise ValueError(f"Unknown scenario: {name}")

results = {}

for scenario in WEIGHT_SCENARIOS:
    grid.at_node["topographic__elevation"][:] = dem_ll.ravel()

    wb = make_weight_builder(scenario, rainfall_ll, ndvi_ll, grid)

    ic = ConnectivityIndex(
        grid,
        flow_director     = FLOW_DIRECTOR,
        weight            = wb,
        w_min             = W_MIN,
        use_degree_approx = USE_DEG_APPROX,
    )

    t0 = time.time()
    ic.run_one_step()
    elapsed = time.time() - t0

    IC_2d = ic.as_2d()
    valid = IC_2d[np.isfinite(IC_2d)]
    results[scenario] = {
        "IC": IC_2d,
        "W": ic.as_2d("connectivity_index__W"),
        "S": ic.as_2d("connectivity_index__S"),
        "Dup": ic.as_2d("connectivity_index__Dup"),
        "Ddn": ic.as_2d("connectivity_index__Ddn"),
        "median": float(np.median(valid)),
        "min": float(valid.min()),
        "max": float(valid.max()),
        "elapsed_s": float(elapsed),
    }

summary_rows = [
    {
        "scenario": k,
        "IC_min": v["min"],
        "IC_max": v["max"],
        "IC_median": v["median"],
        "runtime_s": v["elapsed_s"],
    }
    for k, v in results.items()
]

import pandas as pd
summary_df = pd.DataFrame(summary_rows).sort_values("scenario")
display(summary_df.round(4))

## 5 · Visualise

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5), dpi=120)
if n == 1:
    axes = [axes]

fig.suptitle("IC maps across weight scenarios", fontweight="bold", fontsize=12)

for ax, (name, payload) in zip(axes, results.items()):
    IC_2d = payload["IC"]
    vmin = np.nanpercentile(IC_2d, 2)
    vmax = np.nanpercentile(IC_2d, 98)
    im = ax.imshow(IC_2d, cmap="RdYlGn", vmin=vmin, vmax=vmax, interpolation="nearest", aspect="equal")
    ax.set_title(name.replace("_", " "), fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8, label="IC")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "IC_weight_scenarios.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved ->", OUTPUT_DIR / "IC_weight_scenarios.png")

In [ ]:
IC_final = results[PLOT_SCENARIO]["IC"]

fig, (ax_map, ax_hist) = plt.subplots(
    1, 2, figsize=(12, 5), dpi=130, gridspec_kw={"width_ratios": [2, 1]}
 )

vmin = np.nanpercentile(IC_final, 2)
vmax = np.nanpercentile(IC_final, 98)
im = ax_map.imshow(
    IC_final, cmap="RdYlGn", vmin=vmin, vmax=vmax, interpolation="nearest", aspect="equal"
 )
ax_map.set_title(
    f"IC — scenario: {PLOT_SCENARIO}\n{DEM_SOURCE} + {RF_SOURCE} + {NDVI_SOURCE}\n"
    f"{START_DATE} to {END_DATE}, flow: {FLOW_DIRECTOR}",
    fontsize=9,
 )
ax_map.axis("off")
plt.colorbar(im, ax=ax_map, label="IC (log10)", shrink=0.85)

vals = IC_final[np.isfinite(IC_final)]
ax_hist.hist(vals, bins=60, color="steelblue", edgecolor="none", alpha=0.85)
ax_hist.axvline(
    np.median(vals), color="tomato", lw=1.8, ls="--", label=f"Median = {np.median(vals):.2f}"
)
ax_hist.set_xlabel("IC")
ax_hist.set_ylabel("Cell count")
ax_hist.set_title("IC distribution")
ax_hist.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "IC_outlet_final.png", dpi=130, bbox_inches="tight")
plt.show()

## 6 · Export to GeoTIFF

In [ ]:
def save_geotiff(arr_2d, path, profile):
    out = arr_2d.astype(np.float32)
    out[~np.isfinite(out)] = -9999.0
    p = profile.copy()
    p.update(dtype="float32", count=1, nodata=-9999.0, compress="lzw")
    with rasterio.open(path, "w", **p) as dst:
        dst.write(out, 1)
    print(f"  OK {path}")

payload = results[PLOT_SCENARIO]
save_geotiff(payload["IC"], OUTPUT_DIR / f"IC_outlet_{PLOT_SCENARIO}.tif", profile)
save_geotiff(payload["W"], OUTPUT_DIR / f"W_{PLOT_SCENARIO}.tif", profile)
save_geotiff(payload["S"], OUTPUT_DIR / f"S_{PLOT_SCENARIO}.tif", profile)
save_geotiff(payload["Dup"], OUTPUT_DIR / f"Dup_{PLOT_SCENARIO}.tif", profile)
save_geotiff(payload["Ddn"], OUTPUT_DIR / f"Ddn_{PLOT_SCENARIO}.tif", profile)

summary_df.to_csv(OUTPUT_DIR / "weight_scenarios_summary.csv", index=False)
print("\nAll outputs saved to:", OUTPUT_DIR.resolve())

## 7 · Interpretation

| IC range | Connectivity class | Physical meaning |
|---|---|---|
| IC > 0 | High | Cell is well-connected; sediment and water can easily reach the outlet |
| -1 < IC < 0 | Moderate | Intermediate connectivity |
| IC < -1 | Low | Cell is effectively disconnected; significant storage / impedance |

### Weight scenario interpretation
- `roughness_only`: DEM-only analysis (no NDVI/rainfall needed).
- `rainfall_only`: hydro-climatic control only.
- `ndvi_only`: vegetation/cover control only.
- `ndvi_rainfall`: default hydrological weighting.
- `ndvi_rainfall_roughness`: combined forcing + terrain roughness.

This lets users run IC with DEM only, NDVI only, rainfall only, or any combination depending on data availability.